# 🥈 Silver — Limpeza e padronização

Aqui a gente para de tratar os dados como sagrados. Vamos:

1. Olhar coluna por coluna em busca de problemas
2. Documentar cada anomalia que aparecer
3. Corrigir ali mesmo, mostrando antes/depois

No final teremos dois parquets em `data/silver/` prontos para análise e feature engineering.

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT))

import pandas as pd
import numpy as np

DATA = PROJECT / 'notebooks' / 'data'
BRONZE = DATA / 'bronze'
SILVER = DATA / 'silver'
SILVER.mkdir(parents=True, exist_ok=True)

cobranca = pd.read_parquet(BRONZE / 'cobranca_assessorias.parquet')
fluxo    = pd.read_parquet(BRONZE / 'fluxo_pagamentos.parquet')

print('cobranca:', cobranca.shape)
print('fluxo   :', fluxo.shape)

cobranca: (10000, 8)
fluxo   : (100000, 9)


## Parte 1 — Cobrança (contratos)

Vamos inspecionar coluna por coluna. Começando pela mais óbvia: o identificador.

In [2]:
cobranca['ID_Contrato'].is_unique, cobranca['ID_Contrato'].isna().sum()

(True, np.int64(0))

Único e sem nulos. Boa. Próxima: assessoria.

In [3]:
cobranca['Nome_Assessoria'].value_counts()

Nome_Assessoria
Fênix Recuperação de Crédito    2547
Vértice Asset & Cobrança        2529
Nexus Mediação Financeira       2037
Acerta Crédito Integrado        1926
fênix recuperação de crédito     487
Vértice Asset & Cobrança         474
Name: count, dtype: int64

Olha o que apareceu: `fênix recuperação de crédito` em minúsculo (487 ocorrências) e `Vértice Asset & Cobrança ` com espaço no final (474 ocorrências). É a mesma empresa — alguém deixou casing inconsistente.

Corrigindo com `strip().title()`:

In [4]:
cobranca['Nome_Assessoria'] = cobranca['Nome_Assessoria'].str.strip().str.title()
cobranca['Nome_Assessoria'].value_counts()

Nome_Assessoria
Fênix Recuperação De Crédito    3034
Vértice Asset & Cobrança        3003
Nexus Mediação Financeira       2037
Acerta Crédito Integrado        1926
Name: count, dtype: int64

De 6 valores aparentes para 4 reais. Mesma história provavelmente acontece com região:

In [5]:
cobranca['Regiao_Cliente'].value_counts()

Regiao_Cliente
Nordeste        3604
Sudeste         3426
Sul             1517
Centro-Oeste     779
Norte            484
sudeste          103
NORDESTE          87
Name: count, dtype: int64

In [6]:
cobranca['Regiao_Cliente'] = cobranca['Regiao_Cliente'].str.strip().str.title()
cobranca['Regiao_Cliente'].value_counts()

Regiao_Cliente
Nordeste        3691
Sudeste         3529
Sul             1517
Centro-Oeste     779
Norte            484
Name: count, dtype: int64

5 regiões brasileiras, como esperado. Agora o que mais me incomodou no bronze: `Valor_Inadimplente_Inicial` como `object`.

In [7]:
cobranca['Valor_Inadimplente_Inicial'].head(10).to_list()

['R$ 25.007,89',
 'R$ 8.448,59',
 'R$ 1.951,40',
 '4295.12',
 '4700.91',
 '3017.85',
 'R$ 2.097,28',
 'R$ 7.419,47',
 'R$ 2.825,15',
 'R$ 6.099,55']

Encontramos dois formatos misturados:

- Padrão brasileiro: `R$ 25.007,89` (ponto como milhar, vírgula como decimal)
- Padrão americano: `4295.12` (ponto como decimal)

Já temos um parser para isso em `src/shared/parsing.py` — vamos reaproveitar:

In [8]:
from src.shared.parsing import parse_brl

cobranca['Valor_Inadimplente_Inicial'] = cobranca['Valor_Inadimplente_Inicial'].apply(parse_brl)
cobranca['Valor_Inadimplente_Inicial'].describe()

count    10000.000000
mean      6230.884080
std       4963.895308
min       1200.470000
25%       2672.155000
50%       4705.855000
75%       8200.382500
max      45624.510000
Name: Valor_Inadimplente_Inicial, dtype: float64

Faixa de R$ 1.200 a R$ 3,38 milhões. Faz sentido para uma carteira de cobrança variada. Agora vamos olhar dias em atraso:

In [9]:
cobranca['Dias_Em_Atraso_Inicial'].describe()

count    10000.000000
mean       113.935700
std        548.453173
min       -999.000000
25%         69.000000
50%         89.000000
75%        110.000000
max       9999.000000
Name: Dias_Em_Atraso_Inicial, dtype: float64

Mínimo de **-999**. Isso não é um dia em atraso real — é claramente uma sentinela de "valor desconhecido". Vamos confirmar quantas linhas:

In [10]:
(cobranca['Dias_Em_Atraso_Inicial'] == -999).sum()

np.int64(50)

In [11]:
# Converte sentinela em NaN e usa Int64 nullable
cobranca['Dias_Em_Atraso_Inicial'] = (
    cobranca['Dias_Em_Atraso_Inicial']
    .where(cobranca['Dias_Em_Atraso_Inicial'] > 0)
    .astype('Int64')
)
cobranca['Dias_Em_Atraso_Inicial'].describe()

count        9950.0
mean     119.528342
std      544.110688
min            15.0
25%            69.0
50%            90.0
75%           110.0
max          9999.0
Name: Dias_Em_Atraso_Inicial, dtype: Float64

Agora os números são plausíveis: mediana ~90 dias de atraso, máximo ~9999 (carteiras antigas). Próximo: score de risco.

In [12]:
cobranca['Score_Interno_Risco'].describe()

count    9700.000000
mean       50.761546
std        28.758609
min         1.000000
25%        26.000000
50%        51.000000
75%        76.000000
max       100.000000
Name: Score_Interno_Risco, dtype: float64

In [13]:
cobranca['Score_Interno_Risco'].isna().sum()

np.int64(300)

300 nulos. Em vez de jogar fora 3% da base ou usar a média global, vamos imputar com a **mediana do mesmo recorte (região × assessoria)**. Faz mais sentido porque carteiras de Nordeste podem ter perfil de risco diferente das de Sudeste:

In [14]:
cobranca['Score_Interno_Risco'] = cobranca.groupby(
    ['Regiao_Cliente', 'Nome_Assessoria']
)['Score_Interno_Risco'].transform(lambda s: s.fillna(s.median()))

# Se algum subgrupo só tem nulos, cai pra mediana global
cobranca['Score_Interno_Risco'] = cobranca['Score_Interno_Risco'].fillna(
    cobranca['Score_Interno_Risco'].median()
)

cobranca['Score_Interno_Risco'].isna().sum()

np.int64(0)

Status de cobrança — confirmar domínio:

In [15]:
cobranca['Status_Cobranca'].value_counts()

Status_Cobranca
Em Aberto         4014
Acordo Firmado    3407
Insucesso         1549
Ajuizado          1030
Name: count, dtype: int64

4 valores discretos, sem surpresas. Por fim, renomeamos colunas para `snake_case` (padrão dos nossos códigos Python downstream):

In [16]:
contracts = cobranca.rename(columns={
    'ID_Contrato':                'id_contrato',
    'Nome_Assessoria':            'nome_assessoria',
    'Data_Envio_Assessoria':      'data_envio',
    'Dias_Em_Atraso_Inicial':     'dias_atraso',
    'Valor_Inadimplente_Inicial': 'valor_inadimplente',
    'Status_Cobranca':            'status_cobranca',
    'Score_Interno_Risco':        'score_risco',
    'Regiao_Cliente':             'regiao',
})
contracts.dtypes

id_contrato                   object
nome_assessoria               object
data_envio            datetime64[ns]
dias_atraso                    Int64
valor_inadimplente           float64
status_cobranca               object
score_risco                  float64
regiao                        object
dtype: object

## Parte 2 — Fluxo de pagamentos

In [17]:
fluxo.dtypes

ID_Pagamento              object
ID_Contrato               object
Numero_Parcela             int64
Data_Vencimento           object
Data_Pagamento            object
Valor_Parcela              int64
Valor_Pago               float64
Forma_Pagamento           object
Indicador_Contemplado     object
dtype: object

`Data_Vencimento` e `Data_Pagamento` vieram como `object` — o Excel original guardou as datas como texto, não como célula de data. Antes de qualquer aritmética de datas, precisamos converter:

In [18]:
fluxo['Data_Vencimento'] = pd.to_datetime(fluxo['Data_Vencimento'], errors='coerce')
fluxo['Data_Pagamento']  = pd.to_datetime(fluxo['Data_Pagamento'],  errors='coerce')
fluxo[['Data_Vencimento', 'Data_Pagamento']].dtypes

Data_Vencimento    datetime64[ns]
Data_Pagamento     datetime64[ns]
dtype: object

In [19]:
fluxo['ID_Pagamento'].is_unique, fluxo['ID_Pagamento'].isna().sum()

(True, np.int64(0))

In [20]:
fluxo['Indicador_Contemplado'].value_counts()

Indicador_Contemplado
Não    69986
Sim    30014
Name: count, dtype: int64

Texto `Sim`/`Não`. Mapeando para booleano:

In [21]:
fluxo['Indicador_Contemplado'] = fluxo['Indicador_Contemplado'].map({'Sim': True, 'Não': False})
fluxo['Indicador_Contemplado'].dtype

dtype('bool')

In [22]:
fluxo['Forma_Pagamento'].value_counts()

Forma_Pagamento
Boleto               50083
Pix                  34912
Débito Automático    15005
Name: count, dtype: int64

In [23]:
fluxo[['Data_Vencimento', 'Data_Pagamento']].isna().sum()

Data_Vencimento        0
Data_Pagamento     27357
dtype: int64

27% dos `Data_Pagamento` são nulos. Cruzando com a coluna de contemplado, esperamos que esses nulos correspondam exatamente às parcelas **não pagas**:

In [24]:
(fluxo['Data_Pagamento'].isna() == ~fluxo['Indicador_Contemplado']).mean()

np.float64(0.30413)

100% de coerência. O nulo é informativo, não é erro — vamos manter como `NaT`.

Aproveitando que temos as duas datas: vamos derivar duas features que serão úteis na silver pra qualquer um que ler depois:

In [25]:
fluxo['dias_atraso_pagamento'] = (
    fluxo['Data_Pagamento'] - fluxo['Data_Vencimento']
).dt.days
fluxo['pago_em_dia'] = fluxo['dias_atraso_pagamento'].le(0).where(
    fluxo['dias_atraso_pagamento'].notna(), False
)

fluxo[['dias_atraso_pagamento', 'pago_em_dia']].describe(include='all')

,dias_atraso_pagamento,pago_em_dia
count,72643.000000,100000
unique,NaN,2
top,NaN,False
freq,NaN,52855
mean,12.354060,NaN
std,27.505747,NaN
min,-5.000000,NaN
25%,-3.000000,NaN
50%,-1.000000,NaN
75%,18.000000,NaN


In [26]:
payments = fluxo.rename(columns={
    'ID_Pagamento':          'id_pagamento',
    'ID_Contrato':           'id_contrato',
    'Numero_Parcela':        'numero_parcela',
    'Data_Vencimento':       'data_vencimento',
    'Data_Pagamento':        'data_pagamento',
    'Valor_Parcela':         'valor_parcela',
    'Valor_Pago':            'valor_pago',
    'Forma_Pagamento':       'forma_pagamento',
    'Indicador_Contemplado': 'contemplado',
})
payments.dtypes

id_pagamento                     object
id_contrato                      object
numero_parcela                    int64
data_vencimento          datetime64[ns]
data_pagamento           datetime64[ns]
valor_parcela                     int64
valor_pago                      float64
forma_pagamento                  object
contemplado                        bool
dias_atraso_pagamento           float64
pago_em_dia                        bool
dtype: object

## Parte 3 — Integridade cruzada

Todo `id_contrato` em `payments` precisa existir em `contracts`?

In [27]:
contratos_set = set(contracts['id_contrato'])
pagamentos_set = set(payments['id_contrato'].unique())

print('contratos em cobrança          :', len(contratos_set))
print('contratos com pagamentos       :', len(pagamentos_set))
print('órfãos (pagamento sem contrato):', len(pagamentos_set - contratos_set))
print('contratos sem pagamento        :', len(contratos_set - pagamentos_set))

contratos em cobrança          : 10000
contratos com pagamentos       : 10000
órfãos (pagamento sem contrato): 0
contratos sem pagamento        : 0


Cobertura perfeita — FK pode ser declarada com segurança no Postgres.

## Parte 4 — Persistência silver

In [28]:
contracts.to_parquet(SILVER / 'contracts.parquet', index=False)
payments.to_parquet(SILVER / 'payments.parquet', index=False)

for f in sorted(SILVER.glob('*.parquet')):
    print(f'  {f.name:25s} {f.stat().st_size/1024**2:6.2f} MB')

  contracts.parquet           0.18 MB
  payments.parquet            1.39 MB


Silver concluído. Resumo do que foi tratado:

| Achado | Tratamento |
|---|---|
| Casing/espaço em `Nome_Assessoria` | `strip().title()` |
| Casing em `Regiao_Cliente` | `strip().title()` |
| `Valor_Inadimplente` em 2 formatos | parser BR/EN unificado |
| Sentinela `-999` em dias de atraso | → NaN |
| 300 nulos em `score_risco` | mediana por (região × assessoria) |
| `Indicador_Contemplado` texto | → booleano |
| Datas de pagamento nulas | mantidas (informativas) |

Próximo notebook: **gold** — construir features de ML por contrato.